# Reproducing Wirtshafter and Wilson (2020) from CRCNS hc-29

Paper. Wirtshafter HS, Wilson MA (2020). Differences in reward biased spatial representations in
the lateral septum and hippocampus. *eLife* 9:e55252.
[doi:10.7554/eLife.55252](https://doi.org/10.7554/eLife.55252). The claims file
(`claims/wirtshafter-2020.yaml`) was written from the version of record (PMC7274787), its Methods
and the legends of all figures and figure supplements, and from the paper's archived code
(github.com/elifesciences-publications/data_analysis at commit 5c6320a9).

Dataset. CRCNS hc-29, "Tetrode recordings of hippocampus CA1 and dorsal lateral septum in rat"
([doi:10.6080/K0NG4NV8](https://doi.org/10.6080/K0NG4NV8)), read from the public bucket
`crcnsarchive` under `hc-29/live/`. CRCNS data sets are not versioned; the version is recorded as
`unversioned`. The paper cites the data set by this DOI and no version. The only published content
is the one in `live/` (its description is version 0.6, April 20, 2020, the initial version; there
is no `old/` folder), so this is the version pinned. The MD5 sums of the files read, from the data
set's `checksums.md5`, are printed and checked below against the bytes read:
HPC_clusters.mat `9c1c78340f110cec1a5236d6c898e363`, LS_clusters.mat
`1c092144ac0a4c7f005279f7c50dc694`, Times.mat `8683ffcfc54a43b13e1473ea694f104e`.

What the data set holds. Sorted spike times of 178 CA1 and 452 lateral septum (LS) clusters,
and, per recording, a column of sample timestamps (most at 0.512 ms intervals). It holds no position, speed,
direction, lap, trial or epoch data. Every analysis in the paper beyond counting units uses the
animal's position on the maze (rate maps, place fields, direction of travel, location on the
maze, laps, the matching of CA1 and LS fields for the cross-correlations). So every claim but
the first is blocked, `data_missing`; the unit counts that the first claim gives are computed and
shown, and the data are checked.

| Claim | Outcome |
|---|---|
| `units-recorded` | partially_reproduced (the two counts of cells meeting the criteria blocked: data_missing) |
| `fig2ab-fraction-spatial` | blocked: data_missing |
| `fig2c-bits-per-spike` | blocked: data_missing |
| `fig2b-poisson-control` | blocked: data_missing |
| `fig2s1-mutual-information` | blocked: data_missing |
| `fig2f-bits-per-second` | blocked: data_missing |
| `fig3ab-field-counts` | blocked: data_missing |
| `fig3-field-number-distribution` | blocked: data_missing |
| `fig3s1-speed-acceleration` | blocked: data_missing |
| `place-cell-abundance` | blocked: data_missing |
| `fig4ab-field-length` | blocked: data_missing |
| `fig4ef-lap-accuracy-change` | blocked: data_missing |
| `fig4ef-hpc-more-accurate` | blocked: data_missing |
| `fig4gh-firing-increases` | blocked: data_missing |
| `fig5abc-skew-direction` | blocked: data_missing |
| `fig5s4-uni-bidirectional` | blocked: data_missing |
| `fig5s5-skew-frai` | blocked: data_missing |
| `fig5e-reward-proximate-skew` | blocked: data_missing |
| `fig5s2-skew-distance` | blocked: data_missing |
| `fig6-location-counts` | blocked: data_missing |
| `fig6de-choice-side-bias` | blocked: data_missing |
| `fig6-direction-split` | blocked: data_missing |
| `fig6g-ls-choice-by-direction` | blocked: data_missing |
| `fig6s2-rate-not-distance` | blocked: data_missing |
| `fig6hi-probability-distance` | blocked: data_missing |
| `fig7a-crosscorr-choice` | blocked: data_missing |
| `fig7b-lag` | blocked: data_missing |
| `fig7s2-ls-rates-by-location` | blocked: data_missing |
| `fig7s3-pair-distance-control` | blocked: data_missing |
| `fig7s1-above-shuffle` | blocked: data_missing |
| `fig7c-thirds` | blocked: data_missing |

`units-recorded` is partially reproduced: the two counts of units recorded (452 LS, 178 CA1) match
the clusters deposited exactly, and the other two (cells meeting the rate and track coverage
criteria) need positions and are untested.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

In [ ]:
import re
import time
import resource

import numpy as np
import pandas as pd
import yaml

import hc29

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 80)

started = time.time()
units, spikes, raw = hc29.load_units()
print(f"loaded {len(units)} clusters from {list(hc29.FILES.values())} in {time.time() - started:.1f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
listing = hc29.list_bucket()
print("Objects under hc-29/ in the bucket:")
print(listing.to_string(index=False))
sums = hc29.checksums()
print("\nchecksums.md5:", sums)
for region, fname in hc29.FILES.items():
    import hashlib
    got = hashlib.md5(raw[region]).hexdigest()
    print(f"{fname}: md5 of bytes read {got}, listed {sums['data/' + fname]}, match {got == sums['data/' + fname]}")
t0 = time.time()
times_md5, times_bytes = hc29.stream_md5(hc29.TIMES)
print(f"{hc29.TIMES}: md5 of the {times_bytes} bytes streamed {times_md5}, listed {sums['data/' + hc29.TIMES]}, "
      f"match {times_md5 == sums['data/' + hc29.TIMES]} ({time.time() - t0:.0f} s)")

The bucket holds three data files, the example script `spiketrain.m` and the description, as the
description says. The MD5 sums of all three data files match the data set's `checksums.md5`.

In [ ]:
print("Variables in each cluster file and the form of each field:")
print(units.groupby("region").agg(variable=("variable", "first"), clusters=("key", "size"),
                                  field_shapes=("shape", lambda s: sorted(set(s))[:3]),
                                  all_sorted=("sorted", "all"),
                                  min_spikes=("n_spikes", "min"), median_spikes=("n_spikes", "median"),
                                  max_spikes=("n_spikes", "max"),
                                  first_spike_s=("first_s", "min"), last_spike_s=("last_s", "max")).to_string())
print("\nClusters by region and rat:")
print(pd.crosstab(units.rat, units.region, margins=True).to_string())
ls = units[units.region == "LS"]
print("LS clusters in rats 10 and 12, the rats with CA1 clusters:", int(ls.rat.isin(["rat10", "rat12"]).sum()), "of", len(ls))
print("\nRecordings (rat, date, suffix) with clusters, by region:")
print(units.groupby(["region", "rat"]).times_key.nunique().unstack(0).fillna(0).astype(int).to_string())
print("\nTetrode numbers by region:", units.groupby("region").tetrode.agg(["min", "max", "nunique"]).to_dict("index"))
print("\nField names not of the form '..._cl_<n>':", units[~units.standard_form].field.tolist())
print("Cluster labels that are not numbers:", units[~units.cluster.str.fullmatch(r"\d+")].field.tolist())
print("Session suffixes used:", sorted(set(units.suffix)))
print("LS fields with the tag '_maze' after the tetrode:", int(units[units.region == "LS"].maze_tag.sum()),
      "of", int((units.region == "LS").sum()))

The description says the files hold spike times labelled by session date, tetrode and cluster; the
field names carry the rat as well. Spike times are in seconds (they start near 0 and end within
each recording's timestamps, below), and every train is sorted. HPC clusters are on tetrodes 1 to
16 and LS clusters on tetrodes 17 to 32, consistent with one 16-tetrode array per region (Methods,
Tetrode implementation). The LS clusters come from six rats (6, 7, 8, 9, 10, 12), as the paper's
"six male Long Evans rats" says, but the CA1 clusters from two of them only (rats 10 and 12); the
paper does not say how the 178 CA1 units are spread over rats. So any comparison of CA1 with LS
over all units, as in every claim from Figure 2 on, would set the CA1 units of two rats against
LS units of six, of which rats 10 and 12 give 184 of 452 (printed below); a difference between
the regions there could also be a difference between rats. Ten field names of rat 7 lack the
underscore before the cluster number (`..._cl1`), and one LS cluster is labelled `MUA9`, a
multi-unit label, on rat 9; the 452 LS clusters include it.

In [ ]:
layout = hc29.times_layout()
edges = hc29.times_edges(layout.times_key)
rec = layout.merge(edges, on=["times_key", "samples"])
rec["rat"] = rec.times_key.str.extract(r"date_(rat\d+)_")[0]
rec["duration_h"] = (rec.t_last_s - rec.t_first_s) / 3600
per_rec = units.groupby(["times_key", "region"]).size().unstack(fill_value=0)
rec = rec.merge(per_rec, left_on="times_key", right_index=True, how="left").fillna({"HPC": 0, "LS": 0})
print("Times.mat variables:", layout.variable.unique().tolist(), "; fields:", len(layout),
      "; classes:", layout.matlab_class.unique().tolist(), "; dtypes:", layout.dtype.unique().tolist())
print("Recordings per rat:", rec.rat.value_counts().sort_index().to_dict())
rec["interval_ms"] = rec.dt_median_ms.round(4)
print("Recordings by median sample interval (ms), and by rat:")
print(pd.crosstab(rec.rat, rec.interval_ms).to_string())
print("Largest departure of an interval from the median over the first %d samples (ms): %.4f"
      % (hc29.EDGE_SAMPLES, np.maximum(rec.dt_max_ms - rec.dt_median_ms, rec.dt_median_ms - rec.dt_min_ms).max()))
print("Span / ((samples - 1) x median interval): min %.6f, max %.6f" % (rec.span_over_n_dt.min(), rec.span_over_n_dt.max()))
print("Recordings whose first or last samples are not strictly increasing:", rec[~rec.monotonic_edges].times_key.tolist())
print("Recording durations (h): min %.2f, median %.2f, max %.2f" % (rec.duration_h.min(), rec.duration_h.median(), rec.duration_h.max()))
print("Recordings with no cluster in either file:", int(((rec.HPC == 0) & (rec.LS == 0)).sum()), "of", len(rec))
print("Cluster recordings missing from Times.mat:", sorted(set(units.times_key) - set(rec.times_key)))
print()
print(rec[["times_key", "samples", "t_first_s", "t_last_s", "duration_h", "dt_median_ms", "HPC", "LS"]]
      .round(4).to_string(index=False))

Times.mat holds one column of timestamps per recording, 57 recordings of the same six rats, with no
gap: each recording's span equals its number of samples times its median interval, and the
intervals at the start of each recording depart from it by at most 0.104 ms. Most recordings are at 0.512 ms (1953.125 Hz); six
early recordings of rat 6 are at 3.2 ms (312.5 Hz) and one short recording of rat 7 at 0.0512 ms.
The description gives no sampling rate. The last samples of rat 6's 2017-04-26 recording repeat
timestamps. These are time bases of continuous recordings, not positions. The recordings last
from 2.5 min to 31 h, while the paper describes one 30 min session a day on the maze (Results,
opening paragraph; Figure 1C); nothing in the data set marks which part of a recording is the
maze session, and some recording names say "maze_and_sleep". Eleven recordings have no cluster
in either file.

In [ ]:
t0 = time.time()
iv = hc29.times_intervals(rec.times_key)
print(f"Every sample interval of the {len(iv)} recordings, {iv.samples.sum()} samples ({time.time() - t0:.0f} s):")
print("Recordings with more than one median over pieces of %d samples:" % hc29.PIECE,
      iv[iv.piece_medians_ms.map(len) > 1][["times_key", "piece_medians_ms"]].to_dict("records"))
print("Intervals not positive: %d; over 1.5 x the median: %d; below 0.5 x the median: %d"
      % (iv.not_positive.sum(), iv.long.sum(), iv.short.sum()))
print("Longest interval / median, over recordings: max %.3f" % (iv.longest_ms / iv.dt_median_ms).max())
print(iv[["times_key", "samples", "dt_median_ms", "not_positive", "long", "short", "longest_ms", "longest_at_s"]]
      .round(4).to_string(index=False))
odd = iv[(iv.not_positive > 0) | (iv.long > 0) | (iv.short > 0)]
print("\nRecordings with any interval not positive, long or short: where the long intervals start, and the span of the repeats")
print(odd[["times_key", "dt_median_ms", "long_at_s", "not_positive_from_s", "not_positive_to_s"]].round(3).to_string(index=False))


Read whole, every recording keeps one sampling interval from its first sample to its last: the
median over each piece of samples is the same throughout. In the 50 recordings at 0.512 ms no
interval is longer than 0.69 ms, none is shorter than half the median and none repeats. The
departures are elsewhere. The rat 7 recording at 0.0512 ms has 35,069 intervals over 1.5 times
its median, none longer than 0.149 ms, which is jitter of the timestamps at that fine sampling.
The six rat 6 recordings at 3.2 ms have 25 brief gaps of 8.7 to 93 ms at scattered times, and two
of them repeat timestamps: 1121 in 3 s near 28,667 s of 2017-04-21 (which also has four
intervals under half the median) and 3280 at the last timestamp of 2017-04-26. None of these is long enough, or placed so, to mark a 30 min session:
the time bases hold no change of sampling rate, no gap of even a tenth of a second and no
embedded event codes that could mark the maze session.


In [ ]:
groups = hc29.identical_groups(spikes)
copied = {k for g in groups if len(g) > 2 for k in g}
ranges = rec.set_index("times_key")[["t_first_s", "t_last_s"]]
chk = units.join(ranges, on="times_key")
chk["outside"] = [int(np.sum((spikes[k] < a) | (spikes[k] > b))) for k, a, b in zip(chk.key, chk.t_first_s, chk.t_last_s)]
print("Spikes outside their recording's timestamps:", int(chk.outside.sum()), "of", int(chk.n_spikes.sum()),
      "; clusters with any:", int((chk.outside > 0).sum()),
      "; of these, clusters that are copies of a train deposited more than twice:", int(chk[chk.outside > 0].key.isin(copied).sum()))
span = chk.groupby("times_key").agg(rat=("rat", "first"), clusters=("key", "size"), first_spike_s=("first_s", "min"),
                                    last_spike_s=("last_s", "max"), rec_end_s=("t_last_s", "first"))
span["spike_span_h"] = (span.last_spike_s - span.first_spike_s) / 3600
span["rec_h"] = span.rec_end_s / 3600
span["fraction"] = span.spike_span_h / span.rec_h
print("\nHours of each recording spanned by its clusters' spikes, by rat:")
print(span.groupby("rat").agg(recordings=("clusters", "size"), spike_span_h_min=("spike_span_h", "min"),
                              spike_span_h_median=("spike_span_h", "median"), spike_span_h_max=("spike_span_h", "max"),
                              rec_h_median=("rec_h", "median"), fraction_median=("fraction", "median")).round(2).to_string())

All but 5403 of 15.6 million spikes fall within their recording's timestamps, and every spike
outside lies in a copy of the two rat 7 trains deposited many times (below), which do not belong to
most of the recordings they are filed under. So the spike times and the time files cover the same
spans, and the description's statement is consistent with the data. With no position, trial or
event data, the alignment of spikes to behaviour cannot be checked for any recording: there is
nothing to align to. In rats 7, 8 and 9 the spikes span 0.5 to 0.9 h of each recording, in line
with a 30 min session and its margins (rat 7's recordings are about that long, rat 8's and rat 9's
longer); in rat 6 they span 0.5 to 4.9 h; in rats 10 and 12 they span the whole recording, 3.8 to
6.4 h, so those clusters hold spikes from well beyond a 30 min maze session, which nothing marks.

In [ ]:
multi = [k for g in groups if len(g) > 2 for k in g]   # the rat 7 trains filed under many recordings
gp = hc29.spike_gaps(units, spikes, exclude=multi)
print(f"Spike-free stretches of at least {hc29.GAP_S:.0f} s, all clusters of a recording pooled "
      f"(the {len(multi)} copies of the rat 7 trains deposited more than twice left out):", int(gp.gaps.sum()))
print(gp[gp.gaps > 0][["times_key", "gap_starts_s", "gap_lengths_s"]].to_string(index=False))
print("\nLongest spike-free stretch per recording (s), by rat:")
print(gp.assign(rat=gp.times_key.str.extract(r"date_(rat\d+)_")[0]).groupby("rat").longest_gap_s
      .describe()[["count", "min", "50%", "max"]].round(2).to_string())


The verifier asked whether the maze session could be found from the spikes, as a stretch of a
recording set off by silence. Pooling all of a recording's clusters, no recording has a stretch of
60 s or more without a spike; the longest silence anywhere is 38 s (rat 6, 2017-05-01), and in
rats 10 and 12, whose spikes cover recordings of 3.8 to 6.4 h, under 13 s. So the spikes do not
mark a 30 min maze session either, and with no position or event data the session cannot be cut
out of any recording.


In [ ]:
fields = set(units.field) | set(layout.times_key)
words = ["pos", "x", "y", "vel", "speed", "track", "lap", "trial", "epoch", "maze_time", "behav", "camera"]
hits = {w: sorted(f for f in fields if re.search(rf"(^|_){w}(_|$)", f)) for w in words}
print("Variables in the three files:", sorted(set(units.variable) | set(layout.variable)))
print("Field names containing a position or behaviour word as a token:", {w: h for w, h in hits.items() if h})
print("Field names that are not a cluster or a time base:",
      sorted(f for f in fields if not (f.startswith("cluster_") or f.endswith("_time"))))

The files hold two kinds of field only: clusters of spike times and recording time bases. There is
no position, speed, direction, lap or trial variable. The paper's Methods ("The animal's position
during trial runs was collected at 30 Hz via overhead cameras"; Position and velocity sampling)
and every analysis from Figure 2 on use position, and the paper's Data availability names only
this data set. The paper's code repository at the archived commit holds code only (its non-code
files are notes, headers and library examples; the listing below shows them).

In [ ]:
files = hc29.repo_files()
print(f"{hc29.CODE_REPO} at {hc29.CODE_COMMIT}: {len(files)} files, {files['size'].sum() / 1e6:.1f} MB in all")
print("Files by extension:", files.ext.value_counts().to_dict())
print("\nThe 10 largest files:")
print(files.nlargest(10, "size").to_string(index=False))
data_like = files[files.ext.isin([".mat", ".nex", ".csv", ".dat", ".pos", ".p", ".npy", ".h5", ".zip", ".prn", ".txt"])
                  | (files.ext == "(none)")]
print(f"\nFiles that could hold data (.mat, .nex, .csv, .dat, .pos, .p, .npy, .h5, .zip, .prn, .txt, no extension): {len(data_like)}")
print("Of these, outside the third-party libraries under others_analysis_code/ and hannah-in-use/include/:")
own = data_like[~data_like.path.str.startswith(("others_analysis_code/", "hannah-in-use/include/"))]
print(own.to_string(index=False))


The paper's code repository at the archived commit holds about 35 MB of files. The largest are
library sample data (chronux's spike sorting examples, under 1 MB each). Outside the third-party
libraries, the files that are not code are a README, notes, to-do lists and headers (such as
`posheader.txt`, a header describing position files, not positions) of a few kB. A 30 min session
tracked at 30 Hz would be about 54,000 positions, so no session's tracking is in the repository.


In [ ]:
ix = units.set_index("key")
print("Distinct spike trains:", len(spikes) - sum(len(g) - 1 for g in groups), "of", len(spikes), "clusters")
r7 = [k for g in groups if len(g) > 2 for k in g]
print("Recordings holding a copy of the two trains deposited more than twice:", ix.loc[r7].times_key.nunique(),
      "of", units[units.rat == "rat7"].times_key.nunique(), "rat 7 recordings with clusters")
for g in groups:
    sub = ix.loc[g]
    print(f"\n{len(g)} clusters share one train of {sub.n_spikes.iloc[0]} spikes; region {sorted(set(sub.region))}, "
          f"rats {sorted(set(sub.rat))}, recordings {sub.times_key.nunique()}, tetrodes {sorted(set(sub.tetrode))}")
    print("   ", sorted(sub.field)[:6], "...")
dups = hc29.duplicate_pairs(units, spikes)
same = dups[dups.same_recording].copy()
same["region_a"] = same.a.map(ix.region); same["region_b"] = same.b.map(ix.region)
same["tet_a"] = same.a.map(ix.tetrode); same["tet_b"] = same.b.map(ix.tetrode)
same["n_a"] = same.a.map(ix.n_spikes); same["n_b"] = same.b.map(ix.n_spikes)
same["rat"] = same.a.map(ix.rat)
same["same_tetrode"] = same.tet_a == same.tet_b
print(f"\nPairs on one recording where at least {hc29.DUPLICATE_FRACTION:.0%} of the smaller train's spikes "
      f"have a spike of the other within {hc29.COINCIDENCE_S * 1e3:.1f} ms: {len(same)}")
print(same.groupby(["rat", "region_a", "region_b", "same_tetrode"]).size().to_string())
contained = same[same.coincident_fraction >= 0.99]
print(f"\nOf these, {len(contained)} pairs share at least 99% of the smaller train's spikes; the larger clusters involved:")
big = np.where(contained.n_a >= contained.n_b, contained.a, contained.b)
print(pd.Series(big).map(ix.rat).value_counts().to_dict(), "; distinct larger clusters:", len(set(big)),
      "; distinct smaller clusters:", len(set(np.where(contained.n_a >= contained.n_b, contained.b, contained.a))))
cross = same[~same.same_tetrode]
print("\nPairs across tetrodes (tetrode pairs):", cross.groupby(["rat", "tet_a", "tet_b"]).size().to_dict())
partial = cross[~cross.identical]
print("Coincident fraction of the cross-tetrode pairs that are not identical copies: min %.3f, max %.3f, n %d"
      % (partial.coincident_fraction.min(), partial.coincident_fraction.max(), len(partial)))
t26 = units[(units.rat == "rat6") & (units.tetrode == 26)]
holders = {k for k in big if ix.loc[k, "rat"] == "rat6" and ix.loc[k, "tetrode"] == 26}
print("Rat 6 recordings with tetrode 26 clusters:", t26.times_key.nunique(),
      "; with a cluster holding at least 99%% of another's spikes: %d" % ix.loc[sorted(holders)].times_key.nunique())
# chance level for the coincidence measure: a train shifted by 1 s against its partner
shifted = [hc29.coincident_fraction(spikes[a], spikes[b] + 1.0) for a, b in zip(same.a, same.b)]
print("Same pairs, one train shifted by 1 s: median coincident fraction %.3f, max %.3f" % (np.median(shifted), np.max(shifted)))

Two spike trains of rat 7 are each deposited many times: one of 1033 spikes under 18 cluster names
and one of 1441 spikes under 25, byte for byte the same arrays, spread over 16 of the 19 rat 7 recordings with clusters and three
tetrodes (18, 27 and 32); they hold every spike that falls outside its recording (above). Two clusters of rat 6 (2017-05-01, tetrode 26) are also
identical. On rat 6's tetrode 26, in the recordings counted above, a large cluster holds at least
99% of the spikes of the other clusters on the tetrode, as if a cluster of all spikes were
deposited beside the units. Pairs of clusters on neighbouring tetrodes (29 and 30, 21 and 22 in
LS; 1 and 6 in CA1), in rats 7, 9, 10 and 12, share most of their spikes within 0.5 ms (the range
is printed above), so the same unit, or the same noise, was probably sorted on two tetrodes. Shifting one train by 1 s brings the coincidence of the same pairs to the chance
level printed above. The paper's counts of 452 and 178 units include these entries, so they are
counted below as deposited and as distinct trains.

In [ ]:
# the duplicate search on synthetic trains with known answers
a = np.arange(0, 100, 0.1)                      # 1000 spikes
b = np.sort(np.concatenate([a[::2] + 0.0003, np.arange(0.05, 100, 1.0)]))   # half of a within 0.3 ms, plus 100 others
c = a + 0.002                                   # every spike 2 ms off
print("coincident_fraction: a with itself %.3f (1), b with a %.3f (500/600 = 0.833), a with c %.3f (0), empty %.3f (0)"
      % (hc29.coincident_fraction(a, a), hc29.coincident_fraction(a, b), hc29.coincident_fraction(a, c),
         hc29.coincident_fraction(a, np.array([]))))
toy_spikes = {"X:a": a, "X:a2": a.copy(), "X:b": b, "X:c": c, "X:a3": a.copy()}
toy_units = pd.DataFrame({"key": list(toy_spikes), "times_key": ["r1", "r1", "r1", "r1", "r2"]})
print("identical_groups:", hc29.identical_groups(toy_spikes), "(expected one group: a, a2, a3)")
print(hc29.duplicate_pairs(toy_units, toy_spikes).round(3).to_string(index=False))
print("expected: on r1, a-a2 (1, identical), a-b and a2-b (0.833); across recordings a-a3 and a2-a3 (identical); c with none")


The verifier could not run the duplicate search; here it is run on synthetic trains whose
answers are known, and gives them.


## units-recorded

> 452 caudodorsal LS units and 178 CA1 units were recorded in six rats, of which 165 CA1 and 378 LS
> cells met the spike rate and track coverage criteria.

Criterion: counts from the data set's sorted clusters by region; the inclusion criteria are a mean
rate of at least 0.05 Hz above 12 cm/s and full track coverage, under reading (a) the code's
1950-bin occupancy rule or (b) every arm, the stem and both junctions visited at 12 cm/s or more.
Both readings need positions and speed, which the data set does not hold. The counts of recorded
units are computed, and the rate rule is shown over the whole recording as a variant that decides
nothing.

In [ ]:
dur = rec.set_index("times_key").eval("t_last_s - t_first_s")
units["rate_hz"] = units.n_spikes / units.times_key.map(dur)
copies = {k for g in groups for k in g[1:]}   # every copy beyond the first of an identical train
expected = {"LS": 452, "HPC": 178}
tol = {"LS": 45, "HPC": 18}
out = []
for region in ["LS", "HPC"]:
    u = units[units.region == region]
    out.append({"region": region, "clusters deposited": len(u), "paper": expected[region], "tolerance": tol[region],
                "within": abs(len(u) - expected[region]) <= tol[region],
                "distinct trains": int((~u.key.isin(copies)).sum()),
                "rats": u.rat.nunique(), "recordings": u.times_key.nunique(),
                f"rate >= {hc29.MIN_RATE_HZ} Hz over recording": int((u.rate_hz >= hc29.MIN_RATE_HZ).sum())})
res = pd.DataFrame(out)
print(res.to_string(index=False))
print("\nRats with any cluster:", sorted(set(units.rat)), "->", units.rat.nunique())
print("Paper: 165 CA1 and 378 LS cells met the rate and coverage criteria; these need positions and are not computed.")
print("Whole-recording rate (Hz), by region: ", units.groupby("region").rate_hz.describe()[["min", "50%", "max"]].round(3).to_dict("index"))
print("\nBy rat:")
print(pd.crosstab(units.rat, units.region).to_string())

**Result: partially reproduced.** The data set holds 452 LS and 178 CA1 clusters, exactly the
paper's counts, from six rats (CA1 from two of them). Of the 452 LS clusters, 410 are distinct
spike trains, still within the tolerance of 45. The other two counts, 165 CA1 and 378 LS cells
meeting the spike rate and track coverage criteria, need the animal's position and speed, which
the data set does not hold, so they are untested (data_missing). The criterion needs all four
counts; two hold and two cannot be computed, so the claim holds in part. Over whole recordings
423 LS and 166 CA1 clusters pass the 0.05 Hz rate rule, which says nothing about the
speed-filtered rule of the paper.


## The claims that need position

Every other claim rests on rate maps, place fields, direction of travel, maze location, laps or the
matching of CA1 and LS place fields, all built from the animal's tracked position (Methods,
Position and velocity sampling, Bits/spike, Place field analysis, Cross correlation analysis). The
table gives what each needs.

In [ ]:
claims = yaml.safe_load(open("../../claims/wirtshafter-2020.yaml"))["claims"]
need = {
    "units-recorded": "rate above 12 cm/s and track coverage (position, speed), for two of its four counts",
    "fig2ab-fraction-spatial": "rate maps above 12 cm/s (position, speed)",
    "fig2c-bits-per-spike": "rate maps above 12 cm/s",
    "fig2b-poisson-control": "bits/spike of recorded and Poisson units (rate maps)",
    "fig2s1-mutual-information": "rate maps",
    "fig2f-bits-per-second": "rate maps",
    "fig3ab-field-counts": "place fields (rate maps)",
    "fig3-field-number-distribution": "place fields",
    "fig3s1-speed-acceleration": "rate maps, speed and acceleration per bin",
    "place-cell-abundance": "place fields and the coverage rule",
    "fig4ab-field-length": "place fields",
    "fig4ef-lap-accuracy-change": "passes through fields (position, laps)",
    "fig4ef-hpc-more-accurate": "passes through fields",
    "fig4gh-firing-increases": "passes through fields",
    "fig5abc-skew-direction": "directional fields (position, direction of travel)",
    "fig5s4-uni-bidirectional": "directional fields",
    "fig5s5-skew-frai": "directional fields",
    "fig5e-reward-proximate-skew": "directional fields and maze location",
    "fig5s2-skew-distance": "directional fields and distance to reward",
    "fig6-location-counts": "field centres on the maze",
    "fig6de-choice-side-bias": "field centres on the maze",
    "fig6-direction-split": "directional fields",
    "fig6g-ls-choice-by-direction": "directional fields and maze location",
    "fig6s2-rate-not-distance": "directional fields and distance to reward",
    "fig6hi-probability-distance": "field bins and distance to reward",
    "fig7a-crosscorr-choice": "CA1-LS pairs matched by field centre, located on the maze",
    "fig7b-lag": "matched pairs",
    "fig7s2-ls-rates-by-location": "LS fields located on the maze",
    "fig7s3-pair-distance-control": "matched pairs and field centres",
    "fig7s1-above-shuffle": "forced-arm matched pairs",
    "fig7c-thirds": "matched pairs, by location",
}
table = pd.DataFrame([{"claim": c["id"], "needs": need.get(c["id"], ""),
                       "depends_on": ", ".join(c.get("depends_on", []))} for c in claims])
print(table.to_string(index=False))
print("\nClaims in the claims file:", len(table), "; with an entry in the table of needs:", int(table.claim.isin(need).sum()),
      "; claims without one:", table.claim[~table.claim.isin(need)].tolist(), "; entries for no claim:", sorted(set(need) - set(table.claim)))

## Summary

| Claim | Outcome |
|---|---|
| `units-recorded` | partially_reproduced (the two counts of cells meeting the criteria blocked: data_missing) |
| `fig2ab-fraction-spatial` | blocked: data_missing |
| `fig2c-bits-per-spike` | blocked: data_missing |
| `fig2b-poisson-control` | blocked: data_missing |
| `fig2s1-mutual-information` | blocked: data_missing |
| `fig2f-bits-per-second` | blocked: data_missing |
| `fig3ab-field-counts` | blocked: data_missing |
| `fig3-field-number-distribution` | blocked: data_missing |
| `fig3s1-speed-acceleration` | blocked: data_missing |
| `place-cell-abundance` | blocked: data_missing |
| `fig4ab-field-length` | blocked: data_missing |
| `fig4ef-lap-accuracy-change` | blocked: data_missing |
| `fig4ef-hpc-more-accurate` | blocked: data_missing |
| `fig4gh-firing-increases` | blocked: data_missing |
| `fig5abc-skew-direction` | blocked: data_missing |
| `fig5s4-uni-bidirectional` | blocked: data_missing |
| `fig5s5-skew-frai` | blocked: data_missing |
| `fig5e-reward-proximate-skew` | blocked: data_missing |
| `fig5s2-skew-distance` | blocked: data_missing |
| `fig6-location-counts` | blocked: data_missing |
| `fig6de-choice-side-bias` | blocked: data_missing |
| `fig6-direction-split` | blocked: data_missing |
| `fig6g-ls-choice-by-direction` | blocked: data_missing |
| `fig6s2-rate-not-distance` | blocked: data_missing |
| `fig6hi-probability-distance` | blocked: data_missing |
| `fig7a-crosscorr-choice` | blocked: data_missing |
| `fig7b-lag` | blocked: data_missing |
| `fig7s2-ls-rates-by-location` | blocked: data_missing |
| `fig7s3-pair-distance-control` | blocked: data_missing |
| `fig7s1-above-shuffle` | blocked: data_missing |
| `fig7c-thirds` | blocked: data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. None of the paper's analyses could be run: the data
set holds spike times and recording time bases only, and every finding uses the animal's position.
The units recorded were counted from the clusters deposited.

What was found about the data set.

- It holds no position, speed or direction data, nor any trial, lap or epoch marker, so none of the
  paper's 31 findings can be tested; the paper's counts of units recorded (452 LS, 178 CA1) match
  the clusters deposited.
- The time files are continuous time bases, most at 1953.125 Hz, lasting up to 31 h, with no gap
  and no change of sampling rate inside any recording, while the paper analyses one 30 min maze
  session a day; nothing marks the maze session within a recording, and no recording has a
  stretch of a minute or more without a spike that could mark one.
- In rat 7, two spike trains (1033 and 1441 spikes) are deposited under 18 and 25 cluster names
  across 16 of the 19 rat 7 recordings with clusters and three tetrodes; two clusters of rat 6 are identical; on rat 6's tetrode
  26 one cluster per recording holds all the spikes of the others; pairs of clusters on tetrodes
  29 and 30 (LS) and 1 and 6 (CA1) share most of their spikes. The 452 LS clusters hold 410
  distinct trains.
- CA1 clusters come from two of the six rats, so a CA1 against LS comparison over all units sets
  two rats against six; ten field names of rat 7 break the naming pattern
  (`cl1` for `cl_1`), and one LS cluster is a multi-unit cluster (`MUA9`).

In [ ]:
print(f"Notebook ran in {time.time() - started:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")